In [1]:
import numpy as np
import pandas as pd
import config
import matplotlib.pyplot as plt
# from scipy.stats import shapiro
# from scipy.stats import kruskal

from get_data import DataLoader

# Load data
DataLoader = DataLoader(config.DATA_PATH)
df = DataLoader.load_data()

df_2018pop = df[df['year']==2018].copy()
df_2018pop = df_2018pop[['who_region', 'pm25_concentration', 'population']].dropna()

Categories from Assignment 1

In [2]:
df_2018pop.loc[(df_2018pop['pm25_concentration'] >= 30), 'pm25_cats'] = '(6) 30 +'
df_2018pop.loc[(df_2018pop['pm25_concentration'] >= 25) & (df_2018pop['pm25_concentration'] < 30), 'pm25_cats'] = '(5) 25 - 30'
df_2018pop.loc[(df_2018pop['pm25_concentration'] >= 15) & (df_2018pop['pm25_concentration'] < 25), 'pm25_cats'] = '(4) 15 - 25'
df_2018pop.loc[(df_2018pop['pm25_concentration'] >= 10) & (df_2018pop['pm25_concentration'] < 15), 'pm25_cats'] = '(3) 10 - 15'
df_2018pop.loc[(df_2018pop['pm25_concentration'] >=  5) & (df_2018pop['pm25_concentration'] < 10), 'pm25_cats'] =  '(2) 5 - 10'
df_2018pop.loc[(df_2018pop['pm25_concentration'] <   5), 'pm25_cats'] = '(1) 5 -'

df_2018pop['highest_exposure'] = 0
df_2018pop.loc[(df_2018pop['pm25_concentration'] >= 30), 'highest_exposure'] = 1


In [3]:
df_pop = df_2018pop.dropna(subset='population').reset_index(drop=True)
pop = df_pop['population'].astype(int)

bins = list(range(0, pop.max(), 500000))
labels = [f'{bins[i]}-{bins[i+1]}' for i in range(len(bins)-1)]
df_2018pop['pop_binned'] = pd.cut(df_2018pop['population'], bins=bins, labels=labels, right=False)

print(df_2018pop)

      who_region  pm25_concentration  population    pm25_cats  \
5          4_Eur              14.876    247604.0  (3) 10 - 15   
15         4_Eur              10.808    248960.0  (3) 10 - 15   
29         2_Amr               7.000       639.0   (2) 5 - 10   
47         2_Amr               7.000    121279.0   (2) 5 - 10   
56         5_Emr              49.129   1482816.0     (6) 30 +   
...          ...                 ...         ...          ...   
13472      4_Eur              22.679     63813.0  (4) 15 - 25   
13483      4_Eur              10.396    730828.0  (3) 10 - 15   
13580      4_Eur              30.197     56529.0     (6) 30 +   
13598      4_Eur              17.472    140297.0  (4) 15 - 25   
13657      4_Eur              30.818     62456.0     (6) 30 +   

       highest_exposure       pop_binned  
5                     0         0-500000  
15                    0         0-500000  
29                    0         0-500000  
47                    0         0-500000  
56  

In [4]:
# Step 4: Only use the first 3 population size categories

# Step 5: Group and create tables
df_2018pop2 = df_2018pop.groupby(['pop_binned', 'who_region', 'highest_exposure']).size().reset_index(name='count')
df_2018pop2 = df_2018pop2[df_2018pop2['pop_binned'].isin(['0-500000', '500000-1000000', '1000000-1500000'])].reset_index(drop=True)

print(df_2018pop2.head())


  pop_binned who_region  highest_exposure  count
0   0-500000      1_Afr                 0      0
1   0-500000      1_Afr                 1      0
2   0-500000      2_Amr                 0    138
3   0-500000      2_Amr                 1      0
4   0-500000     3_Sear                 0      0


C:\Users\JoleneWium\AppData\Local\Temp\ipykernel_11528\1429200229.py:4: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df_2018pop2 = df_2018pop.groupby(['pop_binned', 'who_region', 'highest_exposure']).size().reset_index(name='count')


In [5]:
df_500000  = df_2018pop2[df_2018pop2['pop_binned']== '0-500000']
df_1000000 = df_2018pop2[df_2018pop2['pop_binned']== '500000-1000000']
df_1500000 = df_2018pop2[df_2018pop2['pop_binned']=='1000000-1500000']

In [11]:
# Step 1: Pivot the data to have highest_exposure = 0 and 1 as columns
pivot_table = df_2018pop2.pivot_table(
    index=['pop_binned', 'who_region'],
    columns='highest_exposure',
    values='count',
    observed=False,
    fill_value=0
).reset_index()

pivot_table['prevalence'] = pivot_table[1] / (pivot_table[0] + pivot_table[1])

print(pivot_table.head())

highest_exposure pop_binned who_region      0    1  prevalence
0                  0-500000      1_Afr    0.0  0.0         NaN
1                  0-500000      2_Amr  138.0  0.0    0.000000
2                  0-500000     3_Sear    0.0  0.0         NaN
3                  0-500000      4_Eur  195.0  7.0    0.034653
4                  0-500000      5_Emr    0.0  0.0         NaN


In [13]:
df_500000  = pivot_table[pivot_table['pop_binned']== '0-500000']
df_1000000 = pivot_table[pivot_table['pop_binned']== '500000-1000000']
df_1500000 = pivot_table[pivot_table['pop_binned']=='1000000-1500000']

# print(df_500000.to_latex(index=False))
# print(df_1000000.to_latex(index=False))
print(df_1500000.to_latex(index=False))

\begin{tabular}{llrrr}
\toprule
pop_binned & who_region & 0 & 1 & prevalence \\
\midrule
1000000-1500000 & 1_Afr & 0.000000 & 0.000000 & NaN \\
1000000-1500000 & 2_Amr & 4.000000 & 0.000000 & 0.000000 \\
1000000-1500000 & 3_Sear & 0.000000 & 0.000000 & NaN \\
1000000-1500000 & 4_Eur & 5.000000 & 1.000000 & 0.166667 \\
1000000-1500000 & 5_Emr & 0.000000 & 1.000000 & 1.000000 \\
1000000-1500000 & 7_NonMS & 0.000000 & 0.000000 & NaN \\
\bottomrule
\end{tabular}

